# Sol-Gel Synthesis Dataset — Tutorial

This notebook gives a quick introduction to the post-processed sol-gel synthesis dataset.

Each record in the dataset is a single synthesis recipe extracted from a scientific paper and contains three top-level sections:

| Key | Contents |
|---|---|
| `recipe` | DOI, target material (parsed), metal precursors (parsed), reagents (with PubChem data) |
| `operations` | Ordered list of synthesis steps with parsed time, temperature, pH, ratios |
| `phase_purity` | Phase classification, impurity phases (parsed), characterization methods (raw + normalized) |


In [1]:
import json

DATASET_FILE = "sol_gel_dataset_05_06_26.jsonl"

records = []
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

# --- Summary ---
classifications = [r["phase_purity"]["classification"] for r in records]
from collections import Counter
cls_counts = Counter(classifications)

print(f"Dataset file:        {DATASET_FILE}")
print(f"Total records:       {len(records):,}")
print()
print("Phase purity breakdown:")
for label, count in cls_counts.most_common():
    print(f"  {label:<20} {count:>7,}  ({count/len(records)*100:.1f}%)")
print()
print("Top-level keys in each record:", list(records[0].keys()))
print("recipe keys:         ", list(records[0]["recipe"].keys()))
print("phase_purity keys:   ", list(records[0]["phase_purity"].keys()))
print("Sample operation keys:", list(records[0]["operations"][0].keys()) if records[0]["operations"] else "none")


Dataset file:        sol_gel_dataset_05_06_26.jsonl
Total records:       148,064

Phase purity breakdown:
  Pure                 112,610  (76.1%)
  Impure                27,970  (18.9%)
  Insufficient info      7,484  (5.1%)

Top-level keys in each record: ['recipe', 'operations', 'phase_purity']
recipe keys:          ['doi', 'target', 'metal_precursors', 'reagents']
phase_purity keys:    ['impurity_phase', 'phase_purity_details', 'classification', 'characterization_methods']
Sample operation keys: ['operation_type', 'operation_details', 'time', 'temperature', 'pH', 'ratios', 'atmosphere']


In [2]:
# Pretty-print one full record
print(json.dumps(records[0], indent=2, ensure_ascii=False))


{
  "recipe": {
    "doi": "10.1016/j.ceramint.2013.07.044",
    "target": {
      "material_string": "SrTiO3",
      "material_form": "powder",
      "material_formula": "SrTiO3",
      "composition": [
        {
          "formula": "SrTiO3",
          "amount": "1",
          "elements": {
            "Sr": "1",
            "Ti": "1",
            "O": "3"
          },
          "species": {
            "Sr": "1",
            "TiO3": "1"
          }
        }
      ],
      "additives": [],
      "is_mixture": false,
      "phase": "",
      "oxygen_deficiency": "no",
      "oxidation_states": {
        "Sr": [
          2
        ],
        "Ti": [
          4
        ],
        "O": [
          -2
        ]
      }
    },
    "metal_precursors": [
      {
        "material_string": "Sr(NO3)2",
        "material_form": "",
        "material_formula": "Sr(NO3)2",
        "composition": [
          {
            "formula": "Sr(NO3)2",
            "amount": "1",
            "elements":

In [3]:
import json
from collections import Counter
from pathlib import Path


stats = {kind: Counter() for kind in ("target", "metal_precursor", "impurity_phase")}
samples_pop = {kind: [] for kind in stats}
samples_emp = {kind: [] for kind in stats}

def tally(entity, kind, n_samples=4):
    ox = entity.get("oxidation_states", None)
    if ox is None:
        stats[kind]["missing"] += 1
        return
    if ox:                       # non-empty dict
        stats[kind]["populated"] += 1
        if len(samples_pop[kind]) < n_samples:
            samples_pop[kind].append((entity.get("material_string", ""), ox))
    else:                        # {}
        stats[kind]["empty"] += 1
        if len(samples_emp[kind]) < n_samples:
            samples_emp[kind].append(
                (entity.get("material_string", ""), entity.get("material_formula", ""))
            )

with open(DATASET_FILE) as f:
    for line in f:
        rec = json.loads(line)
        tally(rec["recipe"]["target"], "target")
        for p in rec["recipe"]["metal_precursors"]:
            tally(p, "metal_precursor")
        for imp in rec["phase_purity"]["impurity_phase"]:
            tally(imp, "impurity_phase")

# Per-entity table
print(f"{'Entity':<18} {'Populated':>10} {'Empty':>8} {'Missing':>8} {'Total':>8}  {'Hit %':>7}")
print("-" * 64)
totals = Counter()
for kind, c in stats.items():
    total = c["populated"] + c["empty"] + c["missing"]
    pct = 100 * c["populated"] / total if total else 0.0
    print(f"{kind:<18} {c['populated']:>10} {c['empty']:>8} {c['missing']:>8} {total:>8}  {pct:>6.1f}%")
    totals.update(c)

g_total = totals["populated"] + totals["empty"] + totals["missing"]
g_pct = 100 * totals["populated"] / g_total if g_total else 0.0
print("-" * 64)
print(f"{'OVERALL':<18} {totals['populated']:>10} {totals['empty']:>8} {totals['missing']:>8} {g_total:>8}  {g_pct:>6.1f}%")

# Examples
print("\n--- Sample POPULATED entries ---")
for kind, items in samples_pop.items():
    print(f"\n[{kind}]")
    for name, ox in items:
        print(f"  {name:32s} -> {ox}")

print("\n--- Sample EMPTY entries (likely doped, organic, or unparseable) ---")
for kind, items in samples_emp.items():
    print(f"\n[{kind}]")
    for name, formula in items:
        print(f"  name={name!r}  material_formula={formula!r}")

Entity              Populated    Empty  Missing    Total    Hit %
----------------------------------------------------------------
target                  72277    75787        0   148064    48.8%
metal_precursor        312205    28874        0   341079    91.5%
impurity_phase          27045     9861        0    36906    73.3%
----------------------------------------------------------------
OVERALL                411527   114522        0   526049    78.2%

--- Sample POPULATED entries ---

[target]
  SrTiO3                           -> {'Sr': [2], 'Ti': [4], 'O': [-2]}
  Sr2TiO4                          -> {'Sr': [2], 'Ti': [4], 'O': [-2]}
  Sr3Ti2O7                         -> {'Sr': [2], 'Ti': [4], 'O': [-2]}
  Sr4Ti3O10                        -> {'Sr': [2], 'Ti': [4], 'O': [-2]}

[metal_precursor]
  Sr(NO3)2                         -> {'Sr': [2], 'N': [5], 'O': [-2]}
  Ti(OC4H9)4                       -> {'Ti': [4], 'O': [-2], 'C': [-2], 'H': [1]}
  Sr(NO3)2                         -

In [5]:
import sys
sys.path.insert(0, ".")  # if the notebook lives next to normalization.py
from normalization import predict_oxidation_states

cases = {
    # Simple inorganic
    "Sr(NO3)2":      {"Sr": [2], "N": [5], "O": [-2]},
    "LiFePO4":       {"Li": [1], "Fe": [2], "P": [5], "O": [-2]},
    "SrTiO3":        {"Sr": [2], "Ti": [4], "O": [-2]},
    # Mixed valence (floor/ceil split expected)
    "Fe3O4":         {"Fe": [2, 3], "O": [-2]},
    "Mn3O4":         {"Mn": [2, 3], "O": [-2]},
    "Co3O4":         {"Co": [2, 3], "O": [-2]},
    # Hydrate — should predict on anhydrous salt only
    "La(NO3)3·6H2O": {"La": [3], "N": [5], "O": [-2]},
    "Fe(NO3)3·9H2O": {"Fe": [3], "N": [5], "O": [-2]},
    # Doped / fractional stoichiometry — must be skipped
    "La0.7Sr0.3MnO3":  {},
    "Li1.5La0.5TiO3":  {},
    # Variable stoichiometry — must fail to {}
    "La1-xSrxMnO3":    {},
    # Empty
    "":                {},
}

print(f"{'Formula':<22} {'Result':<60} {'Status'}")
print("-" * 95)
all_ok = True
for f, expected in cases.items():
    got = predict_oxidation_states(f)
    ok = got == expected
    all_ok &= ok
    status = "OK" if ok else "FAIL"
    print(f"{f:<22} {str(got):<60} {status}")
    if not ok:
        print(f"{'':<22} expected: {expected}")

print("\n" + ("All cases passed." if all_ok else "Some cases failed — review above."))

Formula                Result                                                       Status
-----------------------------------------------------------------------------------------------
Sr(NO3)2               {'Sr': [2], 'N': [5], 'O': [-2]}                             OK
LiFePO4                {'Li': [1], 'Fe': [2], 'P': [5], 'O': [-2]}                  OK
SrTiO3                 {'Sr': [2], 'Ti': [4], 'O': [-2]}                            OK
Fe3O4                  {'Fe': [2, 3], 'O': [-2]}                                    OK
Mn3O4                  {'Mn': [2, 3], 'O': [-2]}                                    OK
Co3O4                  {'Co': [2, 3], 'O': [-2]}                                    OK
La(NO3)3·6H2O          {'La': [3], 'N': [5], 'O': [-2]}                             OK
Fe(NO3)3·9H2O          {'Fe': [3], 'N': [5], 'O': [-2]}                             OK
La0.7Sr0.3MnO3         {}                                                           OK
Li1.5La0.5TiO3         {}     

In [6]:
import json
from collections import defaultdict, Counter


TM_3D_ORDER = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn"]
TM_3D = set(TM_3D_ORDER)

def entity_elements(entity):
    """Set of elements present in a parsed material entity."""
    elems = set()
    for comp in entity.get("composition", []) or []:
        elems.update(comp.get("elements", {}).keys())
    return elems

results = defaultdict(Counter)
mismatch_examples = defaultdict(list)

with open(DATASET_FILE) as f:
    for line in f:
        rec = json.loads(line)
        target = rec["recipe"]["target"]

        # Filter 1: ternary (exactly 3 distinct elements in target)
        elems = entity_elements(target)
        if len(elems) != 3:
            continue
        # Filter 2: contains a 3d TM
        tms = elems & TM_3D
        if not tms:
            continue
        # Filter 3: classification is Pure or Impure (ignore "Insufficient info")
        classification = rec["phase_purity"].get("classification", "")
        if classification not in ("Pure", "Impure"):
            continue

        target_ox = target.get("oxidation_states", {})
        impurities = rec["phase_purity"].get("impurity_phase", [])

        for tm in tms:
            if classification == "Pure":
                results[tm]["pure"] += 1
                continue

            results[tm]["impure"] += 1
            target_states = set(target_ox.get(tm, []))

            # impurity phases that contain this same TM
            tm_imps = [imp for imp in impurities if tm in entity_elements(imp)]
            if not tm_imps:
                continue
            results[tm]["impure_with_tm_imp"] += 1

            if not target_states:
                # Target prediction empty (e.g. doped target) — can't compare
                results[tm]["impure_target_unknown"] += 1
                continue

            for imp in tm_imps:
                imp_states = set(imp.get("oxidation_states", {}).get(tm, []))
                if imp_states and (imp_states - target_states):
                    results[tm]["impure_with_mismatch"] += 1
                    if len(mismatch_examples[tm]) < 3:
                        mismatch_examples[tm].append({
                            "doi": rec["recipe"]["doi"],
                            "target": target.get("material_formula", ""),
                            "target_ox": sorted(target_states),
                            "impurity": imp.get("material_formula", ""),
                            "imp_ox": sorted(imp_states),
                        })
                    break  # one mismatch per recipe per TM is enough

# --- Print the master table ---
header = (f"{'TM':<4} {'#class.':>8} {'#pure':>7} {'#imp':>6} {'%pure':>7}  "
          f"{'#imp_w/TMimp':>13} {'#mismatch':>10} {'%mis_of_imp':>12}")
print(header)
print("-" * len(header))
for tm in TM_3D_ORDER:
    c = results[tm]
    n = c["pure"] + c["impure"]
    if n == 0:
        continue
    pct_pure = 100 * c["pure"] / n
    pct_mis  = (100 * c["impure_with_mismatch"] / c["impure"]) if c["impure"] else 0.0
    print(f"{tm:<4} {n:>8} {c['pure']:>7} {c['impure']:>6} {pct_pure:>6.1f}%  "
          f"{c['impure_with_tm_imp']:>13} {c['impure_with_mismatch']:>10} {pct_mis:>11.1f}%")

# --- Example mismatches: concrete cases for the manuscript ---
print("\n--- Example oxidation-state mismatches (target vs impurity for same TM) ---")
for tm in TM_3D_ORDER:
    for ex in mismatch_examples[tm]:
        print(f"[{tm}] {ex['doi']}")
        print(f"     target   {ex['target']:25s}  {tm}={ex['target_ox']}")
        print(f"     impurity {ex['impurity']:25s}  {tm}={ex['imp_ox']}")

TM    #class.   #pure   #imp   %pure   #imp_w/TMimp  #mismatch  %mis_of_imp
---------------------------------------------------------------------------
Sc        128      94     34   73.4%             14          0         0.0%
Ti       5144    3976   1168   77.3%            614         19         1.6%
V        1220    1012    208   83.0%            129         15         7.2%
Cr        785     657    128   83.7%             84         15        11.7%
Mn       2660    2114    546   79.5%            327        106        19.4%
Fe       8273    6399   1874   77.3%           1283        105         5.6%
Co       3464    2821    643   81.4%            320        159        24.7%
Ni       2772    2211    561   79.8%            332        111        19.8%
Cu       1509    1068    441   70.8%            311         56        12.7%
Zn       3410    2739    671   80.3%            342          0         0.0%

--- Example oxidation-state mismatches (target vs impurity for same TM) ---
[Ti] 10.101

In [8]:
import json
from collections import defaultdict, Counter

OUTPUT_FILE =  DATASET_FILE
MIN_RECIPES_PER_TARGET = 10   # match Fig 3 methodology — drop rare targets

TM_3D_ORDER = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn"]
TM_3D = set(TM_3D_ORDER)

def entity_elements(entity):
    elems = set()
    for comp in entity.get("composition", []) or []:
        elems.update(comp.get("elements", {}).keys())
    return elems

def target_key(target):
    return (target.get("material_formula") or target.get("material_string") or "").strip()

per_target = {}
target_examples = defaultdict(list)

with open(OUTPUT_FILE) as f:
    for line in f:
        rec = json.loads(line)
        target = rec["recipe"]["target"]
        elems = entity_elements(target)
        if len(elems) != 3:
            continue
        tms = elems & TM_3D
        if not tms:
            continue
        classification = rec["phase_purity"].get("classification", "")
        if classification not in ("Pure", "Impure"):
            continue

        key = target_key(target)
        if not key:
            continue

        if key not in per_target:
            per_target[key] = {
                "tms": tms,
                "target_ox": target.get("oxidation_states", {}),
                "n_pure": 0,
                "n_impure": 0,
                "n_imp_with_mismatch": Counter(),
            }
        t = per_target[key]

        if classification == "Pure":
            t["n_pure"] += 1
            continue

        t["n_impure"] += 1
        impurities = rec["phase_purity"].get("impurity_phase", [])
        for tm in tms:
            target_states = set(t["target_ox"].get(tm, []))
            if not target_states:
                continue
            tm_imps = [imp for imp in impurities if tm in entity_elements(imp)]
            for imp in tm_imps:
                imp_states = set(imp.get("oxidation_states", {}).get(tm, []))
                if imp_states and (imp_states - target_states):
                    t["n_imp_with_mismatch"][tm] += 1
                    if len(target_examples[tm]) < 3:
                        target_examples[tm].append((key, imp.get("material_formula", ""),
                                                    sorted(target_states), sorted(imp_states)))
                    break

# --- Apply >= N threshold, then macro-average per TM ---
pure_rates = defaultdict(list)
mismatch_rates = defaultdict(list)
n_eligible = Counter()
n_eligible_with_impure = Counter()
n_excluded = Counter()

for key, t in per_target.items():
    n = t["n_pure"] + t["n_impure"]
    if n == 0:
        continue
    if n < MIN_RECIPES_PER_TARGET:
        for tm in t["tms"]:
            n_excluded[tm] += 1
        continue
    pr = t["n_pure"] / n
    for tm in t["tms"]:
        n_eligible[tm] += 1
        pure_rates[tm].append(pr)
        if t["n_impure"] > 0:
            n_eligible_with_impure[tm] += 1
            mismatch_rates[tm].append(t["n_imp_with_mismatch"][tm] / t["n_impure"])

# --- Print table ---
print(f"Ternary 3d-TM targets; only targets with >= {MIN_RECIPES_PER_TARGET} "
      f"recipes are kept in the macro-average.\n")
header = (f"{'TM':<4} {'#elig':>6} {'#excl':>6} {'<%pure>':>9}  "
          f"{'#elig_w_imp':>13} {'<%mismatch_of_imp>':>20}")
print(header)
print("-" * len(header))
for tm in TM_3D_ORDER:
    n = n_eligible[tm]
    if n == 0 and n_excluded[tm] == 0:
        continue
    avg_pure = 100 * sum(pure_rates[tm]) / n if n else 0.0
    n_imp = n_eligible_with_impure[tm]
    avg_mis = 100 * sum(mismatch_rates[tm]) / n_imp if n_imp else 0.0
    print(f"{tm:<4} {n:>6} {n_excluded[tm]:>6} {avg_pure:>8.1f}%  "
          f"{n_imp:>13} {avg_mis:>19.1f}%")

print("\n--- Sample mismatches (target / impurity / TM oxidation states) ---")
for tm in TM_3D_ORDER:
    for tgt, imp, t_ox, i_ox in target_examples[tm]:
        print(f"[{tm}]  target={tgt:25s}  impurity={imp:20s}  {tm}: {t_ox} -> {i_ox}")

Ternary 3d-TM targets; only targets with >= 10 recipes are kept in the macro-average.

TM    #elig  #excl   <%pure>    #elig_w_imp   <%mismatch_of_imp>
----------------------------------------------------------------
Sc        2     46     57.5%              2                 0.0%
Ti       64    736     75.2%             61                 1.3%
V        17    235     84.2%             17                 4.0%
Cr       17    180     87.2%             12                19.3%
Mn       42    503     77.8%             41                20.0%
Fe       59    692     77.8%             55                 7.5%
Co       34    488     79.3%             32                25.5%
Ni       24    554     80.6%             22                14.5%
Cu       22    357     73.0%             20                30.2%
Zn       51    756     80.4%             46                 0.0%

--- Sample mismatches (target / impurity / TM oxidation states) ---
[Ti]  target=La2Ti2O7                   impurity=LaTiO3         